Data Extraction and Transformation


What was I working on:

getting service_ids and calendar dates set, add day of week to kcm database for reference 

search how many routes are per service id/how many service ids particular routes have for future reference and to decide what service id to use in calculations 


In [176]:
import sqlite3 as sql
import numpy as np, pandas as pd 
import matplotlib.pyplot as plt 
from datasets import load_dataset



def create_connection(db_file):
    ###create a database connection to the SQLite database specified by db_file
    conn = None
    try: 
        conn = sql.connect(db_file)
        return conn
    except sql.Error as e:
        print(e)

conn = create_connection(r"kcmetro_data.db")


In [153]:
#close database connection 
conn.close()

Merge all kcmetro data needed in order to calculate frequency based on time criteria, save to kcmetro_data.db

In [154]:
trips = pd.read_csv(r'..\data\KCM_data\trips.txt')
stops = pd.read_csv(r'..\data\KCM_data\stops.txt')
stop_times = pd.read_csv(r'..\data\KCM_data\stop_times.txt')
routes = pd.read_csv(r'..\data\KCM_data\routes.txt')

##merge all dataframes together

kcm = trips.merge(stop_times, how='left', on = 'trip_id').merge(routes, how='left', on='route_id').merge(stops, how='left', on = 'stop_id')


###drop irrelevant data not needed to calculate frequency of transit service
drop_cols = ['trip_short_name', 
             'block_id', 
             'shape_id', 
             'peak_flag', 
             'fare_id', 
             'wheelchair_accessible', 
             'bikes_allowed', 
             'arrival_time', 
             'stop_headsign', 
             'agency_id', 
             'shape_dist_traveled', 
             'pickup_type',
             'timepoint',
             'route_long_name',
             'route_desc',
             'route_url',
             'route_color',
             'route_type',
             'route_text_color',
             'tts_stop_name',
             'stop_lat',
             'stop_lon',
             'location_type',
             'stop_url',
             'parent_station',
             'wheelchair_boarding',
             'zone_id',
             'drop_off_type',
             'stop_timezone',
             'stop_desc'
             ]
kcm = kcm.drop(columns=drop_cols)




In [155]:
import re
###split up routes by type, a-h and 1-99 are city, 100-200 is south, 200-300 is eastside, 300-400 is shoreline/north kc, one line and two line, streetcars, and water taxi 
services = {}

services['rapid_ride'] = kcm.loc[kcm.route_short_name.str.contains('Line')]
services['city'] = kcm.loc[kcm.route_short_name.str.contains(r'^\d{1,2}$')]
services['south'] = kcm.loc[kcm.route_short_name.str.contains(r'^1\d{2}$')]
services['east'] = kcm.loc[kcm.route_short_name.str.contains(r'^2\d{2}$')]
services['north'] = kcm.loc[kcm.route_short_name.str.contains(r'^3\d{2}$')]
services['st'] = kcm.loc[kcm.route_short_name.str.contains(r'^5\d{2}$')]


In [175]:
###investigate service ids, drop service ids with very short or only a few day long services 
calendar = pd.read_csv(r'../data/KCM_data/calendar.txt')
calendar_dates = pd.read_csv(r'../data/KCM_data/calendar_dates.txt')

##convert to datetime 
calendar.start_date = pd.to_datetime(calendar.start_date, format=r'%Y%m%d')
calendar.end_date = pd.to_datetime(calendar.end_date, format=r'%Y%m%d')

###calculate services to drop based on the service lasting 15 days or longer 
serv_to_drop = (calendar.loc[(calendar.end_date - calendar.start_date) <= pd.to_timedelta(15, unit='days')].service_id.unique()).astype(int)

#create new dataframe to compare 
kcm_new = kcm.drop(kcm.loc[kcm.service_id.isin(serv_to_drop)].index)
print('Before: \n{}\nAfter:\n{}\nTotal Dropped:\n{}'.format(kcm.shape, kcm_new.shape, kcm.shape[0] - kcm_new.shape[0] ))

###edit main kcm dataframe
kcm = kcm_new

Before: 
(1086407, 11)
After:
(1068322, 11)
Total Dropped:
18085


In [157]:


conn = create_connection(r"..\data\KCM_data\kcmetro_data.db")

c = conn.cursor()

###create KCM table in database 
c.execute('''CREATE TABLE IF NOT EXISTS kcm (
                route_id TEXT,
                service_id TEXT,
                trip_id TEXT,
                trip_headsign TEXT,
                direction_id INTEGER,
                shape_id TEXT,
                stop_id TEXT,
                stop_sequence INTEGER,
                arrival_time TEXT,
                departure_time TEXT,
                stop_headsign TEXT,
                pickup_type INTEGER,
                drop_off_type INTEGER
            )''')

###insert data into KCM table
print('\nkcm data columns:\n', kcm.columns.to_list())
kcm.to_sql('kcm', conn, if_exists='replace', index=False)
print(c.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall())

###commit changes and close connection
conn.commit()
conn.close()


kcm data columns:
 ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'departure_time', 'stop_id', 'stop_sequence', 'route_short_name', 'stop_code', 'stop_name']
[('kcm',)]


Create function to calculate frequencies based on criteria: start_time, end_time, day_type 



Pull route shape data and filtered routes based on freq data, and map the filtered routes.

Display Filtered Routes on Map, perhaps use ArcGIS with selected routes 